# 🌊 EcoHack: Карта пластика
### Детекция скоплений пластика по Sentinel-2 · Индекс FDI

**Инструкция:** Запустите все ячейки (Run All), заполните параметры и нажмите «Анализировать».

---
**Метод:** Floating Debris Index (FDI) — Biermann et al. 2020
**Данные:** Sentinel-2 L2A via Microsoft Planetary Computer (бесплатно)

In [ ]:
# Установка зависимостей (если не установлены)
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import planetary_computer
except ImportError:
    install('planetary-computer')

try:
    import pystac_client
except ImportError:
    install('pystac-client')

try:
    import stackstac
except ImportError:
    install('stackstac')

print('✅ Зависимости готовы')

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import folium
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a1628'
matplotlib.rcParams['text.color'] = 'white'

from core.processor import run_pipeline
from viz.maps import make_folium_map
from viz.plots import make_static_png

print('✅ Модули загружены')

In [ ]:
# ─── Интерфейс управления ────────────────────────────────────────

style = {'description_width': '160px'}
layout = widgets.Layout(width='380px')

# Пресеты
presets = {
    'Тихоокеанское мусорное пятно': (28.5, -145.0),
    'Индийский океан': (-20.0, 80.0),
    'Саргассово море': (27.0, -65.0),
    'Средиземное море': (37.0, 15.0),
    '— Ввести вручную —': None,
}

w_preset = widgets.Dropdown(
    options=list(presets.keys()),
    value='Тихоокеанское мусорное пятно',
    description='Район:',
    style=style, layout=layout
)

w_lat = widgets.FloatText(
    value=28.5, step=0.1,
    description='Широта (°N):',
    style=style, layout=layout
)

w_lon = widgets.FloatText(
    value=-145.0, step=0.1,
    description='Долгота (°E):',
    style=style, layout=layout
)

w_days = widgets.IntSlider(
    value=3, min=1, max=30, step=1,
    description='Дней назад:',
    style=style, layout=layout
)

w_buffer = widgets.FloatSlider(
    value=0.5, min=0.1, max=2.0, step=0.1,
    description='Радиус (°):',
    style=style, layout=layout
)

w_cloud = widgets.IntSlider(
    value=85, min=10, max=100, step=5,
    description='Макс. облачность %:',
    style=style, layout=layout
)

btn_run = widgets.Button(
    description='🔍 Анализировать',
    button_style='primary',
    layout=widgets.Layout(width='380px', height='40px')
)

w_progress = widgets.IntProgress(
    value=0, min=0, max=100,
    description='Прогресс:',
    bar_style='info',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='380px')
)

w_status = widgets.Label(value='Ожидание...')
out_map = widgets.Output()
out_plot = widgets.Output()
out_stats = widgets.Output()

def on_preset_change(change):
    coords = presets.get(change['new'])
    if coords:
        w_lat.value, w_lon.value = coords

w_preset.observe(on_preset_change, names='value')

def on_run(btn):
    with out_map:
        clear_output()
    with out_plot:
        clear_output()
    with out_stats:
        clear_output()

    w_progress.value = 0
    w_progress.bar_style = 'info'

    def progress_cb(msg, pct):
        w_progress.value = pct
        w_status.value = msg

    result = run_pipeline(
        lat=w_lat.value,
        lon=w_lon.value,
        days_back=w_days.value,
        buffer=w_buffer.value,
        max_cloud_cover=w_cloud.value,
        progress_cb=progress_cb,
    )

    if not result.success:
        w_progress.bar_style = 'danger'
        w_status.value = '❌ Снимки не найдены. Измените параметры.'
        return

    w_progress.bar_style = 'success'

    # Stats
    with out_stats:
        stats = result.stats
        display(HTML(f"""
        <div style="background:#112240;padding:12px;border-radius:8px;font-family:monospace">
        <b style="color:#4fc3f7">📊 РЕЗУЛЬТАТЫ</b><br><br>
        🛰️ Снимков найдено: <b>{result.scenes_found}</b><br>
        📅 Даты: <b>{', '.join(result.scene_dates)}</b><br>
        🔴 Покрытие пластиком: <b>{stats.get('plastic_coverage_pct',0):.3f}%</b><br>
        ☁️ Облачность: <b>{stats.get('cloud_coverage_pct','?')}%</b><br>
        📈 FDI макс: <b>{stats.get('fdi_max', 0):.4f}</b><br>
        ⏱️ Время: <b>{result.processing_time_sec} с</b>
        </div>
        """))

    # Folium map
    with out_map:
        m = make_folium_map(
            lat=result.lat, lon=result.lon,
            fdi=result.fdi, plastic_mask=result.plastic_mask,
            lons=result.lons, lats=result.lats,
            cloud_mask=result.cloud_mask,
            stats=result.stats, scene_dates=result.scene_dates,
        )
        display(m)

    # Static plot
    with out_plot:
        import io
        from IPython.display import Image as IPImage
        png = make_static_png(
            fdi=result.fdi, plastic_mask=result.plastic_mask,
            lons=result.lons, lats=result.lats,
            lat=result.lat, lon=result.lon,
            cloud_mask=result.cloud_mask,
            stats=result.stats, scene_dates=result.scene_dates,
        )
        display(IPImage(data=png, width=800))

btn_run.on_click(on_run)

# Layout
controls = widgets.VBox([
    widgets.HTML('<h3 style="color:#4fc3f7">⚙️ Параметры</h3>'),
    w_preset, w_lat, w_lon,
    widgets.HTML('<hr style="border-color:#333">'),
    w_days, w_buffer, w_cloud,
    widgets.HTML('<hr style="border-color:#333">'),
    btn_run, w_progress, w_status,
    widgets.HTML('<hr style="border-color:#333">'),
    out_stats,
], layout=widgets.Layout(width='420px', padding='10px'))

results = widgets.VBox([
    widgets.HTML('<h3 style="color:#4fc3f7">🗺️ Интерактивная карта</h3>'),
    out_map,
    widgets.HTML('<h3 style="color:#4fc3f7">📊 Статический график</h3>'),
    out_plot,
])

display(widgets.HBox([controls, results]))

---
## 📚 Справочная информация

### Индекс FDI (Floating Debris Index)

$$FDI = B_8 - \left[B_6 + (B_{11} - B_6) \cdot \frac{\lambda_8 - \lambda_6}{\lambda_{11} - \lambda_6} \cdot 10\right]$$

| Канал | Длина волны | Роль |
|-------|-------------|------|
| B6    | 740 нм      | Red Edge 2 — базовая линия |
| B8    | 842 нм      | NIR — целевой канал |
| B11   | 1610 нм     | SWIR 1 — базовая линия |

**Почему FDI работает?**  
Пластик имеет характерное поглощение в SWIR и повышенное отражение в NIR по сравнению с чистой водой.  
FDI > 0 сигнализирует о плавающем материале (пластик, водоросли, пена).

### Маскировка облаков
Используется **SCL (Scene Classification Layer)** — стандартная маска классификации Sentinel-2 L2A.  
Маскируются классы: 3 (тень от облаков), 8 (умеренные облака), 9 (плотные облака), 10 (тонкая дымка), 11 (снег).

### Источник данных
**Microsoft Planetary Computer** — бесплатный облачный каталог данных ДЗЗ.  
Sentinel-2 L2A — атмосферно-скорректированные данные, готовые к анализу.

**Биллиография:**  
Biermann et al. (2020). Finding Plastic Patches in Coastal Waters using Optical Satellite Data. *Scientific Reports* 10, 5364. https://doi.org/10.1038/s41598-020-62298-z